## Part 1. Load and clean 

This cell repeats Lab 2's cleaning in one go, so everyone starts from the same table: the word `blank` becomes a real missing value, and pipes shorter than one metre are dropped and counted. Run it and check the numbers against what you found last Friday.

In [1]:
import pandas as pd
import numpy as np
import json

with open("../data/MetroStops.geojson", encoding="utf-8") as f:
    data = json.load(f)

# Turn the GeoJSON properties into a table, like the sewer CSV
df = pd.DataFrame([ft["properties"] for ft in data["features"]])
df.columns = df.columns.str.strip().str.lower() 
print(df.columns.tolist())
print("Stations read:", len(df))

# Clean: keep only metro stations
before = len(df)

print("Dropped (not a station):", before - len(df))

# Clean: drop rows with no coordinates
before = len(df)
coords = [ft["geometry"]["coordinates"] for ft in data["features"]]
df["stop_lon"] = [coord[0] for coord in coords]
df["stop_lat"] = [coord[1] for coord in coords]
df = df.dropna(subset=["stop_lat", "stop_lon"])
print("Dropped (no coordinates):", before - len(df))

# Rename to plain words and make values readable
clean = pd.DataFrame({
    "station": df["station"].astype(str).str.strip(),
    "line": df["line"].astype(str).str.strip(),
    "wheelchair_access": df["wheelchair_access"].astype(str).str.strip(),
    "lat": df["stop_lat"],
    "lon": df["stop_lon"],
})

# Fix acronyms that .title() lowercases
clean["station"] = clean["station"].str.replace("Uqam", "UQAM").str.replace("Oaci", "OACI")

# Check for values the maps didn't recognize
print("Missing values after mapping:")
print(clean.isna().sum())

print("Rows kept:", len(clean))
print("Unique stations:", clean["station"].nunique())
clean.head()

['station', 'line', 'wheelchair_access']
Stations read: 68
Dropped (not a station): 0
Dropped (no coordinates): 0
Missing values after mapping:
station              0
line                 0
wheelchair_access    0
lat                  0
lon                  0
dtype: int64
Rows kept: 68
Unique stations: 68


,station,line,wheelchair_access,lat,lon
0,Angrignon,Green,Yes,45.446397,-73.603293
1,Monk,Green,No,45.451166,-73.593265
2,Jolicoeur,Green,Yes,45.456783,-73.581991
3,Verdun,Green,No,45.459432,-73.571654
4,De L'Église,Green,No,45.461911,-73.567107


## Part 2. One row, by hand

A GeoJSON **feature** is a dictionary with three entries: its `type`, its `geometry` (where it is) and its `properties` (everything else you want to show). You met dictionaries in the Module 3 videos. Here is one row of the table:

In [2]:
row = clean.iloc[0]
row

station              Angrignon
line                     Green
wheelchair_access          Yes
lat                  45.446397
lon                 -73.603293
Name: 0, dtype: object

Now build the feature for that row by hand. Two things to get right:

1. **Coordinates go longitude first, then latitude.** GeoJSON is `[lon, lat]`. Leaflet's `setView` is `[lat, lon]`. Mixing them up puts Vancouver in the Antarctic, which is exactly what happened in the Module 3 demo.
2. **Numbers from pandas need converting** with `float()` before they go into JSON.

Fill in the three `...` below.

In [3]:
feature = {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [float(row["lon"]), float(row["lat"])],        # longitude first, then latitude
    },
    "properties": {
        "station": row["station"],
        "line": row["line"],
        "wheelchair_access": row["wheelchair_access"],
    },
}
print(json.dumps(feature, indent=2,ensure_ascii=False))

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -73.603293,
      45.446397
    ]
  },
  "properties": {
    "station": "Angrignon",
    "line": "Green",
    "wheelchair_access": "Yes"
  }
}


**Check before you continue.** The first coordinate is about (longitude, west is negative) and the second about 49 (latitude). If yours starts with 49, swap them.

## Part 3. Turn it into a function, and test it

You are about to do the same thing 39,916 times. That is what a function is for. Put your dictionary inside `row_to_feature`, so that it takes any row and returns its feature.

Then the test cell checks it on one row you can check by eye. This is the *test* step of the loop: prove a small piece works before you trust it with everything.

In [4]:
def row_to_feature(row):
    """Take one row of the metro station table and return one GeoJSON feature."""
    return {
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [float(row["lon"]), float(row["lat"])],        # longitude first, then latitude
    },
    "properties": {
        "station": row["station"],
        "line": row["line"],
        "wheelchair_access": row["wheelchair_access"],
    },
}
print(json.dumps(feature, indent=2,ensure_ascii=False))   # paste your dictionary from Part 2 here, and it will work for any row

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      -73.603293,
      45.446397
    ]
  },
  "properties": {
    "station": "Angrignon",
    "line": "Green",
    "wheelchair_access": "Yes"
  }
}


In [5]:
# Test on one row. Every line prints True when the function is right.
test = row_to_feature(clean.iloc[0])
lon, lat = test["geometry"]["coordinates"]
print("is a Feature:          ", test["type"] == "Feature")
print("longitude comes first: ", -74 < lon < -73.4)
print("latitude comes second: ", 45.4 < lat < 45.8)
print("has a metro line:      ", test["properties"]["line"] in ["Green", "Orange", "Yellow", "Blue"])

is a Feature:           True
longitude comes first:  True
latitude comes second:  True
has a metro line:       True


## Part 4. Every row, then write the file

Prediction first. The CSV is about 2.6 MB. The GeoJSON repeats every property name on every row. Will the file be **smaller, about the same, or bigger?** Roughly how many MB?

**My prediction:** 

Then fill in the loop. `clean.iterrows()` gives you each row in turn; the `_` holds the row number, which you do not need.

In [6]:
features = []
for _, row in clean.iterrows():
    features.append(row_to_feature(row))
collection = {"type": "FeatureCollection", "features": features}

out_path = "../data/MetroStops.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)
    # separators: no spaces, a smaller file.
    # allow_nan=False: stop with an error if any value is NaN (an empty cell).
    # Browsers cannot read NaN in JSON, so without this line the file writes
    # fine and the map fails later with a confusing message. Fail here instead.

import os
print(len(features), "features written to", out_path)
print(round(os.path.getsize(out_path) / 1_000_000, 1), "MB")

68 features written to ../data/MetroStops.geojson
0.0 MB


## Part 5. Check your own file

Like Lab 2's check cell, this one does not know the answers. It reads your file back from disk (the way the browser will) and checks it against the table and against Vancouver. Run it.

In [7]:
with open("../data/MetroStops.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(clean):
    problems.append(f"The file has {len(feats)} features but the clean table has {len(clean)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-73.98 < x < -73.47 for x in lons):
    problems.append("Some longitudes are outside Vancouver. Are longitude and latitude swapped?")
if feats and not all(45.40 < y < 45.71 for y in lats):
    problems.append("Some latitudes are outside Vancouver. Are longitude and latitude swapped?")
if feats and not all({"station", "line", "wheelchair_access"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing station, line or wheelchair_access.")
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed


## Now go to the site

The rest of the lab happens in `site/app.js` and the browser. Follow the handout from Part 6.

---

## Layer B. Your own data through the same pipeline

Copy the cells you need into a new notebook in your **project** folder (see the handout). The only parts that change are the file name, the column names inside `row_to_feature`, and the numbers in the check cell. Everything else is the same pipeline.